In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
N = 100 # Gear_Ratio

folder_path = r"S:/Nit Durgapur/College 6th Sem/CSIR-CMERI/Friction_Modelling/EXP_1/EXP_1_Data/12_06_2026"
file = r"J_1_12.06.2026_13_05_36.csv"

file_path = os.path.join(folder_path, file)
df = pd.read_csv(file_path)
print(df.shape)
df.head()

In [ ]:
df["q"] = np.deg2rad(df["q1"])
df["v"] = np.deg2rad(df["v1"])
df["tm"] = df["m_t_1"]
df["tj"] = df["jts1"]

In [ ]:
df["tf"] = (N * df["tm"]) - df["tj"]

In [ ]:
df["timestamp"] = pd.to_datetime("2026-06-12 " + df["timestamp"].astype(str))

In [ ]:
plot_df = df[["tm", "tj", "v", "q"]].copy()
scaler = MinMaxScaler()
plot_scaled = scaler.fit_transform(plot_df)
plot_scaled = pd.DataFrame(
    plot_scaled,
    columns=plot_df.columns
)
plt.figure(figsize=(16,6))
plt.plot(
    df["timestamp"],
    plot_scaled["tm"],
    label="Motor Torque"
)
plt.plot(
    df["timestamp"],
    plot_scaled["tj"],
    label="Joint Torque"
)
plt.plot(
    df["timestamp"],
    plot_scaled["v"],
    label="Velocity"
)
plt.plot(
    df["timestamp"],
    plot_scaled["q"],
    label="Position"
)
plt.legend()
plt.xlabel("Timestamp")
plt.ylabel("Normalized Value")
plt.title("Normalized Joint Signals")
plt.grid(True)
plt.show()

In [ ]:
size = len(df)

train_end = int(0.60 * size)
val_end = int(0.80 * size)

train_df = df.iloc[: train_end]
val_df = df.iloc[train_end : val_end]
test_df = df.iloc[val_end :]

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

In [ ]:
def model1(v, Tc, Bv, Vs):
    return (Tc * np.tanh(v/Vs) + Bv * v)

In [ ]:
v_train = train_df["v"].values
tf_train = train_df["tf"].values

p0 = [5, 0.1, 0.01]

params1, _ = curve_fit(
    model1,
    v_train,
    tf_train,
    p0 = p0,
    maxfev = 50000
)

Tc1, Bv1, Vs1 = params1

print("Tc =",Tc1)
print("Bv =",Bv1)
print("Vs =",Vs1)

In [ ]:
v_test = test_df["v"].values
tf_test = test_df["tf"].values

pred1 = model1(v_test, * params1)

In [ ]:
test_time = test_df["timestamp"]

In [ ]:
rmse1 = np.sqrt(mean_squared_error(tf_test, pred1))
mae1 = mean_absolute_error(tf_test, pred1)
r2_1 = r2_score(tf_test, pred1)

print(rmse1)
print(mae1)
print(r2_1)

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(
    test_time,
    tf_test,
    label="Actual Torque",
    linewidth=1
)
plt.plot(
    test_time,
    pred1,
    label="Coulomb + Viscous Prediction",
    linewidth=1
)
plt.xlabel("Time")
plt.ylabel("Torque")
plt.title(f"Coulomb + Viscous Model\nR²={r2_1:.4f}")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
residual1 = tf_test - pred1
plt.figure(figsize=(15,4))
plt.plot(
    test_time,
    residual1
)
plt.axhline(
    0,
    color='red'
)
plt.xlabel("Time")
plt.ylabel("Residual")
plt.title("Coulomb + Viscous Residual")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(7,7))
plt.scatter(
    tf_test, 
    pred1, 
    s=5, 
    alpha=0.3
)
min_val = min(tf_test.min(), pred1.min())
max_val = max(tf_test.max(), pred1.max())
plt.plot(
    [min_val,max_val],
    [min_val,max_val],
    'r', 
    linewidth=2
)
plt.xlabel("Actual Torque")
plt.ylabel("Predicted Torque")
plt.title("Actual vs Predicted Scatter Plot")
plt.grid(True)
plt.show()

In [ ]:
idx = np.argsort(v_test)
plt.figure(figsize=(10,6))
plt.scatter(
    v_test,
    tf_test,
    s=2,
    alpha=0.2,
    label="Measured"
)
plt.plot(
    v_test[idx],
    pred1[idx],
    linewidth=3,
    label="Coulomb + Viscous"
)
plt.xlabel("Velocity (rad/s)")
plt.ylabel("Residual Torque")
plt.title("Friction Torque vs Velocity")
plt.legend()
plt.grid(True)
plt.show()